# Household MPC Experiment

Dedicated paired baseline-versus-income-shock workflow. The implementation lives in `src/experiments/mpc.py`; the historical command remains available through `run_mpc_experiment.py`.

In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.experiments.mpc import run_mpc_experiment
from src.mpc_analysis import MPCFilterConfig, plot_mpc_panel

## Inputs

In [2]:
RUN_MPC = True
COUNTRY = "FRA"
SEEDS = [1,2,3,4,5, 6,7,8,9,10,11,12,13,14,15,16,17,18,19,20, 21,22,23,24,25,26,27,28,29,30]
T_MAX = 30
SHOCK_PERIOD = 20
HORIZON_PERIODS = 4
SHOCK_FRACTION = 0.1
N_JOBS = 4
OUTPUT_DIR = "data/output_data/mpc-experiment"

filters = MPCFilterConfig(
    min_income=1000,
    income_top_quantile=0.99,
    consumption_income_ratio_quantiles=(0.01, 0.99),
    gross_wealth_top_quantile=0.999,
    head_age_min=25,
    head_age_max=75,
)

## Run

In [3]:
outputs = None
if RUN_MPC:
    outputs = run_mpc_experiment(
        seeds=SEEDS,
        t_max=T_MAX,
        shock_period=SHOCK_PERIOD,
        horizon_periods=HORIZON_PERIODS,
        shock_fraction=SHOCK_FRACTION,
        country_iso3=COUNTRY,
        output_dir=OUTPUT_DIR,
        n_jobs=N_JOBS,
        force_rebuild_data=True,
        apply_mpc_filters=True,
        mpc_filter_config=filters,
        stay_in_activity_bracket_only=False,
        stable_effective_labor_state_only=True,
    )
else:
    print("Set RUN_MPC = True to run the experiment.")

{'seed': 1, 'country': 'FRA', 't_max': 30, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/mpc-experiment', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/mpc-experiment/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
 

/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 6 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+14.20%), industry 2 (+0.00%), industry 9 (+1.02%), industry 12 (+2.84%), industry 14 (+4.92%), industry 15 (+1.99%). These sectors keep their unadjusted Orbis markup and price at average cost until unit depreciation falls relative to marginal cost. The percentage is the AC-floor overshoot of the anchor, also recorded as pricing_markup_residual_unreachable_gap.
  self._warn_on_unreachable_markup_anchors()
/Users/andone/Documents/python_projects/INET-consumption/macromodel/agents/firms/func/prices.py:734: RuntimeWarning: Markup residual calibration: 6 sector(s) have an AC floor above their initial-price anchor and cannot reach it at any residual factor: industry 1 (+14.20%), industry 2 (+0.00%), industry 9 (+1.02%), industry 12

## Results and plots

In [4]:
if outputs is not None:
    panel = pd.read_csv(outputs["analysis_dir"] / "household_mpc_panel.csv")
    summary = pd.read_csv(outputs["analysis_dir"] / "household_mpc_summary.csv")
    plot_mpc_panel(
        panel,
        variables=[
            "income",
            "net_wealth",
            "housing_tenure",
            "net_liquid_financial_assets",
            "illiquid_financial_assets",
            "housing_wealth",
        ],
        mpc_columns=["mpc_impact", "cmpc_4q"],
        plot_kind="box_only",
        y_range=(-0.5, 1.1),
        winsor_quantiles=(0, 1),
        # winsor_quantiles=(0.01, 0.99),
        winsor_mode="trim",
        output_dir=outputs["analysis_dir"] / "plots",
        output_format="png",
    )
    summary